# Publication Protocol V2 Final Evaluation

Locked held-out execution notebook for the predefined five-subject test set.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

def resolve_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'fbn_pipeline').exists():
            return candidate
    raise RuntimeError('Project root not found')

PROJECT_ROOT = resolve_project_root()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from fbn_pipeline import run_publication_protocol_v2_final_evaluation
from fbn_pipeline.data import load_dataset
from fbn_pipeline.final_evaluation import (
    ABLATION_ORDER,
    MAIN_MODEL_ORDER,
    PROPOSED_LABEL,
    _paths,
    _validate_baseline_lock,
    _validate_pipeline_lock,
    _validate_self_hashed_payload,
)

print('PROTOCOL MODE = LOCKED_FINAL_EVALUATION')
print('MODEL SELECTION = DISABLED')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')


PROTOCOL MODE = LOCKED_FINAL_EVALUATION
MODEL SELECTION = DISABLED
PROJECT_ROOT = C:\Users\doqua\Desktop\fuzzy-bayesian-cognitive-load\fts-anfis-bns-final


## 2. Validate all locks and manifests

In [2]:
path_map = _paths(PROJECT_ROOT)
comparison_manifest = _validate_self_hashed_payload(path_map['comparison_manifest'], hash_field='manifest_hash', label='Comparison manifest')
statistical_plan = _validate_self_hashed_payload(path_map['statistical_plan'], hash_field='plan_hash', label='Statistical plan')
proposed_lock = _validate_pipeline_lock(path_map['proposed_lock'])
baseline_paths = {
    'Statistical Logistic Regression': path_map['baseline_lock_dir'] / 'stat_logreg.json',
    'Statistical RBF SVM': path_map['baseline_lock_dir'] / 'stat_svm.json',
    'Statistical Random Forest': path_map['baseline_lock_dir'] / 'stat_random_forest.json',
    'Statistical HistGradientBoosting': path_map['baseline_lock_dir'] / 'stat_hist_gradient_boosting.json',
    'Flattened-window Logistic Regression': path_map['baseline_lock_dir'] / 'flat_logreg.json',
}
ablation_paths = {
    'Full FTS-ANFIS-BN': path_map['ablation_lock_dir'] / 'ABLATION_FULL.json',
    'No BN': path_map['ablation_lock_dir'] / 'ABLATION_NO_BN.json',
    'No ANFIS': path_map['ablation_lock_dir'] / 'ABLATION_NO_ANFIS.json',
    'No FTS': path_map['ablation_lock_dir'] / 'ABLATION_NO_FTS.json',
}
baseline_locks = {name: _validate_baseline_lock(path, comparison_manifest['manifest_hash']) for name, path in baseline_paths.items()}
ablation_locks = {name: _validate_pipeline_lock(path) for name, path in ablation_paths.items()}
validation_status = {
    'comparison_manifest_hash': comparison_manifest['manifest_hash'],
    'statistical_plan_hash': statistical_plan['plan_hash'],
    'proposed_lock_hash': proposed_lock['lock_hash'],
    'baseline_lock_hashes': {name: payload['lock_hash'] for name, payload in baseline_locks.items()},
    'ablation_lock_hashes': {name: payload['lock_hash'] for name, payload in ablation_locks.items()},
    'all_selected_using_training_oof_only': all(payload['selected_using'] == 'training_oof_only' for payload in [proposed_lock, *baseline_locks.values(), *ablation_locks.values()]),
    'all_thresholds_from_training_oof_only': all(payload['threshold_source'] == 'training_oof_only' for payload in [proposed_lock, *baseline_locks.values(), *ablation_locks.values()]),
    'all_final_test_flags_false': all(payload['final_test_evaluated'] is False for payload in [proposed_lock, *baseline_locks.values(), *ablation_locks.values()]),
    'proposed_reconstructed_from_explicit_locked_parameters': bool(proposed_lock['config']),
    'no_hyperparameter_override_interface': True,
}
print(json.dumps(validation_status, indent=2, sort_keys=True))


{
  "ablation_lock_hashes": {
    "Full FTS-ANFIS-BN": "d11021ef3e2e3784b2ca408ad685a890211acbb035b77f3101ca161c30762e4e",
    "No ANFIS": "cd94ca90013633d3f4c1b4442b4135d1a7d6ea917c183ebc60919c1679754da7",
    "No BN": "8b41fed9449a035e1d8aefc33a20d9c56d124ff56821c19f2822af2d78b8c339",
    "No FTS": "43c77b83d7dda740e826034af65a3380a63c437d4e5e73270f8c23d2839dd08e"
  },
  "all_final_test_flags_false": true,
  "all_selected_using_training_oof_only": true,
  "all_thresholds_from_training_oof_only": true,
  "baseline_lock_hashes": {
    "Flattened-window Logistic Regression": "c17a9cddadf6182c687d127f8e31f2a07c96b68e45f268e14350f005ae9df50d",
    "Statistical HistGradientBoosting": "50ffb518c5649cc0e6d6e412b011c133c64a5494e8d96d7285f46531b27b6b98",
    "Statistical Logistic Regression": "2797cf9d4683cec2b05e8328154a7a360ffe22526bff1c2772b5d53a945f949e",
    "Statistical RBF SVM": "239293e5f373f75fff5cee0414c4df942a20478fb838c2c59a7af728f45853b2",
    "Statistical Random Forest": "442a263

## 3. Load the fixed train/test subject split

## 4. Print only split integrity information

In [3]:
dataset = load_dataset(PROJECT_ROOT / 'last_30s_segments')
split_integrity = {
    'train_subject_ids': sorted(np.unique(dataset.user_train).tolist()),
    'test_subject_ids': sorted(np.unique(dataset.user_test).tolist()),
    'train_subject_count': int(np.unique(dataset.user_train).size),
    'test_subject_count': int(np.unique(dataset.user_test).size),
    'train_windows': int(dataset.X_train.shape[0]),
    'test_windows': int(dataset.X_test.shape[0]),
    'train_class_counts': {
        'rest': int(np.sum(dataset.y_train == 0)),
        'task': int(np.sum(dataset.y_train == 1)),
    },
    'test_class_counts': {
        'rest': int(np.sum(dataset.y_test == 0)),
        'task': int(np.sum(dataset.y_test == 1)),
    },
    'subject_overlap': sorted(set(np.unique(dataset.user_train).tolist()).intersection(np.unique(dataset.user_test).tolist())),
}
print(json.dumps(split_integrity, indent=2, sort_keys=True))


{
  "subject_overlap": [],
  "test_class_counts": {
    "rest": 95,
    "task": 98
  },
  "test_subject_count": 5,
  "test_subject_ids": [
    "3caqi",
    "6frz4",
    "bd47a",
    "f1gjp",
    "iz3x1"
  ],
  "test_windows": 193,
  "train_class_counts": {
    "rest": 317,
    "task": 315
  },
  "train_subject_count": 18,
  "train_subject_ids": [
    "1mpau",
    "2nxs5",
    "5gpsc",
    "7swyk",
    "8a1ep",
    "b7mrd",
    "c24ur",
    "dkhty",
    "e4gay",
    "ef5rq",
    "f3j25",
    "hpbxa",
    "ibvx8",
    "iz2ps",
    "rc1in",
    "tn4vl",
    "wjxci",
    "yljm5"
  ],
  "train_windows": 632
}


## 5. Execute the locked proposed model

## 6. Execute all locked baselines

## 7. Execute all locked ablations

In [4]:
bundle = run_publication_protocol_v2_final_evaluation(PROJECT_ROOT)
print(json.dumps({
    'execution_complete': True,
    'main_models_executed': MAIN_MODEL_ORDER,
    'ablations_executed': ABLATION_ORDER,
    'artifacts_created': bundle['created_files'],
}, indent=2))


{
  "execution_complete": true,
  "main_models_executed": [
    "Proposed FTS-ANFIS-BN",
    "Statistical Logistic Regression",
    "Statistical RBF SVM",
    "Statistical Random Forest",
    "Statistical HistGradientBoosting",
    "Flattened-window Logistic Regression"
  ],
  "ablations_executed": [
    "Full FTS-ANFIS-BN",
    "No BN",
    "No ANFIS",
    "No FTS"
  ],
  "artifacts_created": [
    "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\final_evaluation\\final_main_model_metrics.csv",
    "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\final_evaluation\\final_ablation_metrics.csv",
    "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\final_evaluation\\final_per_subject_metrics.csv",
    "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\

## 8. Display the final main-model metrics table

In [5]:
main_table = bundle['main_metrics_df'][['Model', 'Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'TASK F1', 'Macro-F1', 'ROC-AUC']]
print(main_table.to_string(index=False))


                               Model  Accuracy  Balanced Accuracy  Precision  Recall  TASK F1  Macro-F1  ROC-AUC
               Proposed FTS-ANFIS-BN    0.7513             0.7519     0.7778  0.7143   0.7447    0.7511   0.7734
     Statistical Logistic Regression    0.6373             0.6369     0.6373  0.6633   0.6500    0.6368   0.7126
                 Statistical RBF SVM    0.6269             0.6252     0.6102  0.7347   0.6667    0.6216   0.7126
           Statistical Random Forest    0.6891             0.6890     0.6939  0.6939   0.6939    0.6890   0.7126
    Statistical HistGradientBoosting    0.6062             0.6061     0.6122  0.6122   0.6122    0.6061   0.6455
Flattened-window Logistic Regression    0.5492             0.5442     0.5346  0.8673   0.6615    0.4935   0.5454


## 9. Display the ablation metrics table

In [6]:
ablation_table = bundle['ablation_metrics_df'][['Model', 'Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'TASK F1', 'Macro-F1', 'ROC-AUC']]
print(ablation_table.to_string(index=False))


            Model  Accuracy  Balanced Accuracy  Precision  Recall  TASK F1  Macro-F1  ROC-AUC
Full FTS-ANFIS-BN    0.7513             0.7519     0.7778  0.7143   0.7447    0.7511   0.7734
            No BN    0.6891             0.6873     0.6583  0.8061   0.7248    0.6838   0.7158
         No ANFIS    0.5389             0.5385     0.5446  0.5612   0.5528    0.5384   0.5709
           No FTS    0.6477             0.6490     0.6875  0.5612   0.6180    0.6455   0.7241


## 10. Display per-subject results

In [7]:
print(bundle['per_subject_df'].to_string(index=False))


                               Model                                         model_id subject_id  n_samples  Accuracy  Balanced Accuracy  TASK F1  ROC-AUC  roc_auc_defined                                                        lock_hash                                                  prediction_hash
               Proposed FTS-ANFIS-BN PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5      3caqi         39    0.6923             0.6961   0.6471   0.7842             True a2e479262944b50f002d304a2ecb91af6228376b7440ee639ed71169cb80aa37 7bb708563e9baf44045229b1874de10cbc2eef0ecdc0ee6c02f8d669885b7c1b
               Proposed FTS-ANFIS-BN PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5      6frz4         39    0.7179             0.7184   0.7179   0.7237             True a2e479262944b50f002d304a2ecb91af6228376b7440ee639ed71169cb80aa37 7bb708563e9baf44045229b1874de10cbc2eef0ecdc0ee6c02f8d669885b7c1b
               Proposed FTS-ANFIS-BN PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5      bd4

## 11. Compute clustered bootstrap confidence intervals

In [8]:
print(bundle['confidence_df'].to_string(index=False))


                               Model                                         model_id            metric  point_estimate  ci_lower  ci_upper  confidence_level  bootstrap_replicates  valid_replicates  skipped_replicates  seed                                                        lock_hash                                                  prediction_hash
               Proposed FTS-ANFIS-BN PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5          Accuracy          0.7513    0.7047    0.8144              0.95                 10000             10000                   0  4242 a2e479262944b50f002d304a2ecb91af6228376b7440ee639ed71169cb80aa37 7bb708563e9baf44045229b1874de10cbc2eef0ecdc0ee6c02f8d669885b7c1b
               Proposed FTS-ANFIS-BN PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5 Balanced Accuracy          0.7519    0.7052    0.8156              0.95                 10000             10000                   0  4242 a2e479262944b50f002d304a2ecb91af6228376b7440ee639ed71169cb80aa37 7bb

## 12. Run the predefined McNemar comparison

In [9]:
print(json.dumps(bundle['mcnemar_result'], indent=2, sort_keys=True))


{
  "baseline_correct_proposed_wrong": 13,
  "baseline_lock_hash": "2797cf9d4683cec2b05e8328154a7a360ffe22526bff1c2772b5d53a945f949e",
  "baseline_model_id": "STAT_LOGREG__C0.1__CWnone",
  "baseline_prediction_hash": "bc04000cff478debea093512b37f234a2b154ebe717482d55cd31ab977be966b",
  "comparison": "Proposed FTS-ANFIS-BN vs Statistical Logistic Regression",
  "discordant_total": 48,
  "mcnemar_statistic": 9.1875,
  "method": "continuity-corrected",
  "p_value": 0.002437,
  "proposed_correct_baseline_wrong": 35,
  "proposed_lock_hash": "a2e479262944b50f002d304a2ecb91af6228376b7440ee639ed71169cb80aa37",
  "proposed_model_id": "PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5",
  "proposed_prediction_hash": "7bb708563e9baf44045229b1874de10cbc2eef0ecdc0ee6c02f8d669885b7c1b"
}


## 13. Generate final confusion matrices and ROC curves

In [10]:
figure_summary = {
    'confusion_matrix_figure': str((PROJECT_ROOT / 'artifacts' / 'publication_protocol_v2' / 'final_evaluation' / 'final_confusion_matrices.png').resolve()),
    'roc_curve_figure': str((PROJECT_ROOT / 'artifacts' / 'publication_protocol_v2' / 'final_evaluation' / 'final_roc_curves.png').resolve()),
}
print(json.dumps(figure_summary, indent=2, sort_keys=True))


{
  "confusion_matrix_figure": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\final_evaluation\\final_confusion_matrices.png",
  "roc_curve_figure": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\final_evaluation\\final_roc_curves.png"
}


## 14. Run integrity and safeguard checks

In [11]:
print(json.dumps(bundle['integrity_checks'], indent=2, sort_keys=True))


{
  "ablation_full_predictions_identical_to_proposed": true,
  "ablation_full_scientific_config_matches_proposed": true,
  "all_expected_models_evaluated": [
    "PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5",
    "STAT_LOGREG__C0.1__CWnone",
    "STAT_SVM__C0.1__G0.01",
    "STAT_RF__Dnone__L3",
    "STAT_HGB__LR0.05__LN7__L20.0",
    "FLAT_LOGREG__C0.1__CWnone",
    "ABLATION_FULL",
    "ABLATION_NO_BN",
    "ABLATION_NO_ANFIS",
    "ABLATION_NO_FTS"
  ],
  "all_result_rows_record_hashes": true,
  "no_candidate_removed_for_poor_test_performance": true,
  "no_configuration_mutation_after_metrics": true,
  "no_hyperparameter_override_interface": true,
  "test_labels_used_only_for_metrics": true,
  "thresholds_applied_unchanged": true
}


## 15. Print a final JSON-like summary

In [12]:
final_summary = {
    'validation': bundle['validation_summary'],
    'artifact_hashes': bundle['artifact_hashes'],
    'created_files': bundle['created_files'],
    'technical_failures': bundle['technical_failures'],
    'rerun_required': bundle['rerun_required'],
    'closing_statement': 'The held-out evaluation was executed as one locked batch using configurations and thresholds selected exclusively from subject-wise OOF predictions on the 18-subject development set. No model selection or hyperparameter tuning was performed using the five-subject held-out results.',
}
print(json.dumps(final_summary, indent=2, sort_keys=True))


{
  "artifact_hashes": {
    "final_ablation_metrics.csv": "656b79482fae45920ee7a7031a469904bbf76febe64339b0b943a6be50865299",
    "final_confidence_intervals.csv": "04446efac1b89cc32da8186c8dd2794101c1f33f5f5c76a94636b031cdaf59d7",
    "final_confusion_matrices.png": "4e5173c734d5fc1ea8a089f0430441258f0bd8656762695828a04a415b2b0b12",
    "final_evaluation_manifest.json": "eff5f0edd5f3405a7c8adc81f09fae835024f65ee9767d2758da3ab2558b4726",
    "final_main_model_metrics.csv": "afff593340fb493c745205c5faba4dcb0a049ae6c7a88891d9a3a46eff237d0f",
    "final_mcnemar_result.json": "7d22a181e2986cca7aef41663dafc13a341c7f2ac07efe38df87f33c0d5cc2b0",
    "final_per_subject_metrics.csv": "b79e505ca6b50d9cbcf4f8e06567f5750da1c1f8e010d6f56607ae6207673628",
    "final_predictions.npz": "a4007f1231a5b53060f706ef7dbe7d0171d3aa225a5c1966ee8fbffd13412b0d",
    "final_roc_curves.png": "ef440f76749e4501f6386c8a92cbbc516a19cde187fa8cf670a025236069b507"
  },
  "closing_statement": "The held-out evaluation wa